In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

storage      = "stairqualitydev01"
bronze_path  = f"abfss://bronze@{storage}.dfs.core.windows.net/open_meteo_hist/*/*/*/*/*.json"
target_table = "airquality.silver.weather_hourly"

In [0]:
raw = (
    spark.read
         .option("multiLine", True)
         .json(bronze_path)
         .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Files read:", raw.count())
display(raw)

In [0]:
exploded = raw.select(
    F.regexp_extract("source_file", r"open_meteo_hist/([^/]+)/", 1).alias("city"),
    F.col("latitude").alias("grid_latitude"),
    F.col("longitude").alias("grid_longitude"),
    "source_file",
    F.posexplode("hourly.time").alias("pos", "time_str"),
    F.col("hourly.temperature_2m").alias("temps"),
    F.col("hourly.relative_humidity_2m").alias("hums"),
)

display(exploded.limit(5))

In [0]:
silver = (
    exploded.select(
        "city",
        F.to_timestamp("time_str", "yyyy-MM-dd'T'HH:mm").alias("time_utc"),
        F.expr("temps[pos]").cast("double").alias("temperature_c"),
        F.expr("hums[pos]").cast("int").alias("humidity_pct"),
        "grid_latitude",
        "grid_longitude",
        "source_file",
        F.current_timestamp().alias("processed_at"),
    )
    .dropDuplicates(["city", "time_utc"])
)

display(silver.limit(10))

In [0]:
print("Total rows:", silver.count())

display(
    silver.groupBy("city").agg(
        F.count("*").alias("rows"),
        F.min("time_utc").alias("first_hour"),
        F.max("time_utc").alias("last_hour"),
        F.sum(F.col("temperature_c").isNull().cast("int")).alias("missing_temps"),
    )
)

In [0]:
(
    silver.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target_table)
)

print("Saved:", target_table)